In [1]:
from pathlib import Path
import shutil
import random
from tqdm import tqdm
import numpy as np
import librosa
import tensorflow as tf

2026-06-03 20:46:42.668104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780519603.055511      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780519603.177417      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780519604.134479      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780519604.134516      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780519604.134519      58 computation_placer.cc:177] computation placer alr

In [2]:
# =========================
# CONFIG
# =========================

SOURCE_ROOT = Path("/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset")
DEST_ROOT = Path("/kaggle/working/subset_fake_real_dataset_62k")

SEED = 42
random.seed(SEED)

DATASET_PATHS = {
    "for-original": SOURCE_ROOT / "for-original" / "for-original",
    "for-norm": SOURCE_ROOT / "for-norm" / "for-norm",
    "for-rerec": SOURCE_ROOT / "for-rerec" / "for-rerecorded",
    "for-2sec": SOURCE_ROOT / "for-2sec" / "for-2seconds",
}

SPLITS = ["training", "validation", "testing"]
LABELS = ["fake", "real"]

# Jumlah file per dataset-version dan per label
# Training  : 7000 fake + 7000 real per dataset version
# Validation: 500 fake + 500 real per dataset version
# Testing   : 250 fake + 250 real per dataset version
#
# Catatan:
# for-2sec training hanya punya 6978 fake dan 6978 real,
# jadi bagian for-2sec akan otomatis ambil semua file yang tersedia.

SAMPLE_CONFIG = {
    "training": {
        "fake": 7000,
        "real": 7000,
    },
    "validation": {
        "fake": 500,
        "real": 500,
    },
    "testing": {
        "fake": 250,
        "real": 250,
    },
}

In [3]:
def check_dataset_paths(dataset_paths):
    """
    Mengecek apakah semua path dataset sumber tersedia.
    """
    print("Checking dataset paths...\n")
    
    all_ok = True
    
    for dataset_name, dataset_path in dataset_paths.items():
        if dataset_path.exists():
            print(f"[OK] {dataset_name}: {dataset_path}")
        else:
            print(f"[MISSING] {dataset_name}: {dataset_path}")
            all_ok = False
    
    return all_ok

In [4]:
def get_files_from_folder(folder_path):
    """
    Mengambil semua file dari sebuah folder.
    """
    try:
        if not folder_path.exists():
            print(f"[WARNING] Folder tidak ditemukan: {folder_path}")
            return []
        
        files = [file for file in folder_path.iterdir() if file.is_file()]
        return files
    
    except Exception as e:
        print(f"[ERROR] Gagal membaca folder {folder_path}: {e}")
        return []

In [5]:
def sample_files(files, sample_size, dataset_name, split, label):
    """
    Mengambil sample file secara random.
    Jika jumlah file kurang dari kebutuhan, ambil semua file yang tersedia.
    """
    try:
        total_files = len(files)
        
        if total_files == 0:
            print(f"[WARNING] Tidak ada file: {dataset_name} | {split} | {label}")
            return []
        
        if total_files < sample_size:
            print(
                f"[WARNING] File kurang: {dataset_name} | {split} | {label} "
                f"butuh {sample_size}, tersedia {total_files}. Akan ambil semua."
            )
            return files
        
        return random.sample(files, sample_size)
    
    except Exception as e:
        print(f"[ERROR] Gagal sampling {dataset_name} | {split} | {label}: {e}")
        return []

In [6]:
def make_output_dirs(dest_root, splits, labels):
    """
    Membuat struktur folder output:
    training/fake, training/real, validation/fake, dst.
    """
    try:
        for split in splits:
            for label in labels:
                output_dir = dest_root / split / label
                output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"[OK] Folder output siap di: {dest_root}")
    
    except Exception as e:
        print(f"[ERROR] Gagal membuat folder output: {e}")

In [7]:
def copy_sampled_files(sampled_files, dest_dir, dataset_name):
    """
    Copy file sample ke folder tujuan.
    Nama file diberi prefix nama dataset agar tidak bentrok.
    """
    copied_count = 0
    
    for src_file in tqdm(sampled_files, desc=f"Copying {dataset_name}"):
        try:
            new_filename = f"{dataset_name}__{src_file.name}"
            dest_file = dest_dir / new_filename
            
            shutil.copy2(src_file, dest_file)
            copied_count += 1
        
        except Exception as e:
            print(f"[ERROR] Gagal copy {src_file}: {e}")
    
    return copied_count

In [8]:
def create_balanced_subset(
    dataset_paths,
    dest_root,
    sample_config,
    splits,
    labels
):
    """
    Membuat subset dataset seimbang dari beberapa versi dataset.
    """
    print("Starting subset creation...\n")
    
    make_output_dirs(dest_root, splits, labels)
    
    summary = {}
    
    for dataset_name, dataset_path in dataset_paths.items():
        print(f"\n==============================")
        print(f"Processing dataset: {dataset_name}")
        print(f"==============================")
        
        summary[dataset_name] = {}
        
        for split in splits:
            summary[dataset_name][split] = {}
            
            for label in labels:
                src_dir = dataset_path / split / label
                dest_dir = dest_root / split / label
                
                needed_count = sample_config[split][label]
                
                files = get_files_from_folder(src_dir)
                sampled_files = sample_files(
                    files=files,
                    sample_size=needed_count,
                    dataset_name=dataset_name,
                    split=split,
                    label=label
                )
                
                print(
                    f"\n{dataset_name} | {split} | {label}: "
                    f"available={len(files)}, sampled={len(sampled_files)}"
                )
                
                copied_count = copy_sampled_files(
                    sampled_files=sampled_files,
                    dest_dir=dest_dir,
                    dataset_name=dataset_name
                )
                
                summary[dataset_name][split][label] = copied_count
    
    print("\nSubset creation finished.")
    return summary

In [9]:
def count_output_files(dest_root, splits, labels):
    """
    Menghitung jumlah file hasil subset di folder output.
    """
    print("\n==============================")
    print("Final Dataset Count")
    print("==============================")
    
    grand_total = 0
    
    for split in splits:
        print(f"\nSplit: {split}")
        split_total = 0
        
        for label in labels:
            folder = dest_root / split / label
            
            if folder.exists():
                count = len([file for file in folder.iterdir() if file.is_file()])
            else:
                count = 0
            
            split_total += count
            print(f"{label}: {count}")
        
        grand_total += split_total
        print(f"Total {split}: {split_total}")
    
    print(f"\nGrand Total: {grand_total}")

In [10]:
def count_by_dataset_version(dest_root, splits, labels, dataset_names):
    """
    Menghitung jumlah file berdasarkan prefix nama dataset.
    """
    print("\n==============================")
    print("Count by Dataset Version")
    print("==============================")
    
    for split in splits:
        print(f"\nSplit: {split}")
        
        for dataset_name in dataset_names:
            dataset_total = 0
            
            for label in labels:
                folder = dest_root / split / label
                
                if folder.exists():
                    count = len([
                        file for file in folder.iterdir()
                        if file.is_file() and file.name.startswith(f"{dataset_name}__")
                    ])
                else:
                    count = 0
                
                dataset_total += count
                print(f"{dataset_name} | {label}: {count}")
            
            print(f"{dataset_name} total: {dataset_total}\n")

In [11]:
import shutil

if DEST_ROOT.exists():
    shutil.rmtree(DEST_ROOT)
    print(f"Folder lama dihapus: {DEST_ROOT}")

In [12]:
# Cek dulu apakah path dataset tersedia
paths_ok = check_dataset_paths(DATASET_PATHS)

if paths_ok:
    summary = create_balanced_subset(
        dataset_paths=DATASET_PATHS,
        dest_root=DEST_ROOT,
        sample_config=SAMPLE_CONFIG,
        splits=SPLITS,
        labels=LABELS
    )
    
    count_output_files(
        dest_root=DEST_ROOT,
        splits=SPLITS,
        labels=LABELS
    )
    
    count_by_dataset_version(
        dest_root=DEST_ROOT,
        splits=SPLITS,
        labels=LABELS,
        dataset_names=list(DATASET_PATHS.keys())
    )
else:
    print("Ada path dataset yang tidak ditemukan. Cek ulang SOURCE_ROOT atau struktur folder Kaggle input.")

Checking dataset paths...

[OK] for-original: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original
[OK] for-norm: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm
[OK] for-rerec: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-rerec/for-rerecorded
[OK] for-2sec: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-2sec/for-2seconds
Starting subset creation...

[OK] Folder output siap di: /kaggle/working/subset_fake_real_dataset_62k

Processing dataset: for-original

for-original | training | fake: available=26941, sampled=7000


Copying for-original: 100%|██████████| 7000/7000 [00:59<00:00, 117.02it/s]



for-original | training | real: available=26941, sampled=7000


Copying for-original: 100%|██████████| 7000/7000 [01:35<00:00, 73.30it/s]



for-original | validation | fake: available=5400, sampled=500


...


for-2sec | testing | fake: available=544, sampled=250


Copying for-2sec: 100%|██████████| 250/250 [00:01<00:00, 132.89it/s]



for-2sec | testing | real: available=544, sampled=250


Copying for-2sec: 100%|██████████| 250/250 [00:01<00:00, 130.23it/s]



Subset creation finished.

Final Dataset Count

Split: training
fake: 26082
real: 26082
Total training: 52164

Split: validation
fake: 2000
real: 2000
Total validation: 4000

Split: testing
fake: 1000
real: 1000
Total testing: 2000

Grand Total: 58164

Count by Dataset Version

Split: training
for-original | fake: 7000
for-original | real: 7000
for-original total: 14000

for-norm | fake: 7000
for-norm | real: 7000
for-norm total: 14000

for-rerec | fake: 5104
for-rerec | real: 5104
for-rerec total: 10208

for-2sec | fake: 6978
for-2sec | real: 6978
for-2sec total: 13956


Split: validation
for-original | fake: 500
for-original | real: 500
for-original total: 1000

for-norm | fake: 500
for-norm | real: 500
for-norm total: 1000

for-rerec | fake: 500
for-rerec | real: 500
for-rerec total: 1000

for-2sec | fake: 500
for-2sec | real: 500
for-2sec total: 1000


Split: testing
for-original | fake: 250
for-original | real: 250
for-original total: 500

for-norm | fake: 250
for-norm | real: 25

In [13]:
!pip install -q tensorflow-io

In [14]:
from pathlib import Path
import tensorflow as tf

try:
    import tensorflow_io as tfio
    TFIO_AVAILABLE = True
    print("[OK] tensorflow_io tersedia. Resampling aktif.")
except Exception as e:
    TFIO_AVAILABLE = False
    print("[WARNING] tensorflow_io tidak tersedia. Resampling akan dilewati.")
    print(e)

/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZN3tsl7strings13safe_strtou64ESt17basic_string_viewIcSt11char_traitsIcEEPm']
  warnings.warn(

[OK] tensorflow_io tersedia. Resampling aktif.


In [15]:
def load_audio_with_librosa(file_path, target_sample_rate=16000):
    """
    Membaca audio menggunakan librosa agar lebih fleksibel
    terhadap variasi format WAV.
    """
    file_path = file_path.decode("utf-8")

    audio, _ = librosa.load(
        file_path,
        sr=target_sample_rate,
        mono=True
    )

    audio = audio.astype(np.float32)

    return audio

In [16]:
from pathlib import Path

bad_path = Path(
    "/kaggle/working/subset_fake_real_dataset_62k/"
    "training/fake/for-original__file27839.mp3"
)

print("Exists :", bad_path.exists())
print("Is file:", bad_path.is_file())

if bad_path.exists():
    print("Size   :", bad_path.stat().st_size, "bytes")

Exists : True
Is file: True
Size   : 0 bytes


In [17]:
from pathlib import Path
import librosa


def is_audio_readable(file_path, target_sr=16000):
    """
    Mengecek apakah file benar-benar bisa dibaca sebagai audio.
    """
    try:
        audio, _ = librosa.load(
            str(file_path),
            sr=target_sr,
            mono=True,
            duration=0.1
        )

        return len(audio) > 0

    except Exception:
        return False


def find_bad_audio_files(dataset_root):
    """
    Scan seluruh subset dataset dan mengembalikan file rusak.
    """
    dataset_root = Path(dataset_root)
    bad_files = []

    splits = ["training", "validation", "testing"]
    labels = ["fake", "real"]

    for split in splits:
        for label in labels:
            folder = dataset_root / split / label

            print(f"Scanning: {split}/{label}")

            for file_path in folder.iterdir():
                if not file_path.is_file():
                    bad_files.append(file_path)
                    continue

                if file_path.stat().st_size == 0:
                    bad_files.append(file_path)
                    continue

                if not is_audio_readable(file_path):
                    bad_files.append(file_path)

    print("\nScan finished")
    print("=" * 40)
    print(f"Bad files found: {len(bad_files)}")

    return bad_files

In [18]:
bad_files = find_bad_audio_files(
    "/kaggle/working/subset_fake_real_dataset_62k"
)

Scanning: training/fake
Scanning: training/real
Scanning: validation/fake
Scanning: validation/real
Scanning: testing/fake
Scanning: testing/real

Scan finished
Bad files found: 3


In [19]:
def delete_bad_audio_files(bad_files):
    deleted_count = 0

    for file_path in bad_files:
        try:
            if file_path.exists():
                file_path.unlink()
                deleted_count += 1

        except Exception as e:
            print(f"Gagal menghapus {file_path}: {e}")

    print(f"Deleted bad files: {deleted_count}")

In [20]:
delete_bad_audio_files(bad_files)

Deleted bad files: 3


In [21]:
from pathlib import Path
import tensorflow as tf


class PreprocessDatasetMFCCTorchLike:
    def __init__(
        self,
        dataset_root="/kaggle/working/subset_fake_real_dataset_62k",
        sample_rate=16000,
        duration=2.0,
        n_mfcc=40,
        n_mels=64,
        frame_length=512,
        frame_step=160,
        fft_length=512,
        batch_size=32,
        shuffle_buffer=4096,
        ignore_errors=False
    ):
        self.dataset_root = Path(dataset_root)

        self.sample_rate = sample_rate
        self.duration = duration
        self.num_samples = int(sample_rate * duration)

        self.n_mfcc = n_mfcc
        self.n_mels = n_mels

        self.frame_length = frame_length
        self.frame_step = frame_step
        self.fft_length = fft_length

        self.batch_size = batch_size
        self.shuffle_buffer = shuffle_buffer
        self.ignore_errors = ignore_errors

        self.splits = ["training", "validation", "testing"]
        self.labels_map = {
            "real": 0,
            "fake": 1
        }

        self.autotune = tf.data.AUTOTUNE

        self.num_spectrogram_bins = self.fft_length // 2 + 1

        self.mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=self.n_mels,
            num_spectrogram_bins=self.num_spectrogram_bins,
            sample_rate=self.sample_rate,
            lower_edge_hertz=80.0,
            upper_edge_hertz=7600.0
        )

        print("PreprocessDatasetMFCCTorchLike initialized.")
        print(f"Dataset root     : {self.dataset_root}")
        print(f"Sample rate      : {self.sample_rate}")
        print(f"Duration         : {self.duration}")
        print(f"Waveform samples : {self.num_samples}")
        print(f"N MFCC           : {self.n_mfcc}")
        print(f"N Mels           : {self.n_mels}")
        print(f"Frame length     : {self.frame_length}")
        print(f"Frame step       : {self.frame_step}")
        print(f"FFT length       : {self.fft_length}")
        print(f"Batch size       : {self.batch_size}")

    def collect_file_paths(self, split):
        file_paths = []
        labels = []

        split_dir = self.dataset_root / split

        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder tidak ditemukan: {split_dir}")

        for label_name, label_value in self.labels_map.items():
            label_dir = split_dir / label_name

            if not label_dir.exists():
                print(f"[WARNING] Folder tidak ditemukan: {label_dir}")
                continue

            files = sorted([
                str(file)
                for file in label_dir.iterdir()
                if file.is_file()
            ])

            file_paths.extend(files)
            labels.extend([label_value] * len(files))

            print(f"{split} | {label_name}: {len(files)} files")

        print(f"Total {split}: {len(file_paths)} files\n")
        return file_paths, labels

    @tf.function
    @tf.function
    def decode_audio(self, file_path):
        """
        Decode audio memakai librosa melalui tf.numpy_function.
        Output tetap TensorFlow Tensor agar siap masuk pipeline tf.data.
        """
        audio = tf.numpy_function(
            func=lambda path: load_audio_with_librosa(
                path,
                target_sample_rate=self.sample_rate
        ),
            inp=[file_path],
            Tout=tf.float32
        )

        # Panjang masih dinamis sebelum crop/padding
        audio.set_shape([None])

        return audio

    @tf.function
    def fix_audio_length(self, audio):
        audio_length = tf.shape(audio)[0]

        def crop_audio():
            return audio[:self.num_samples]

        def pad_audio():
            padding_amount = self.num_samples - audio_length
            return tf.pad(audio, paddings=[[0, padding_amount]])

        audio = tf.cond(
            audio_length > self.num_samples,
            crop_audio,
            pad_audio
        )

        audio.set_shape([self.num_samples])
        return audio

    @tf.function
    def make_waveform_input(self, audio):
        waveform = tf.expand_dims(audio, axis=-1)
        waveform.set_shape([self.num_samples, 1])
        return waveform

    @tf.function
    def make_mfcc(self, audio):
        """
        MFCC versi lebih dekat ke torchaudio:
        n_fft=512, hop_length=160, n_mels=64.
        """
        # torchaudio MelSpectrogram default center=True,
        # jadi kita pad kiri-kanan sebesar n_fft//2.
        pad = self.fft_length // 2
        audio_centered = tf.pad(audio, paddings=[[pad, pad]])

        stft = tf.signal.stft(
            audio_centered,
            frame_length=self.frame_length,
            frame_step=self.frame_step,
            fft_length=self.fft_length
        )

        spectrogram = tf.abs(stft)
        power_spectrogram = tf.square(spectrogram)

        mel_spectrogram = tf.matmul(
            power_spectrogram,
            self.mel_weight_matrix
        )

        log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

        mfcc = tf.signal.mfccs_from_log_mel_spectrograms(
            log_mel_spectrogram
        )

        mfcc = mfcc[:, :self.n_mfcc]

        # dari (time, mfcc) ke (mfcc, time)
        mfcc = tf.transpose(mfcc)

        # normalisasi seperti PyTorch kamu
        mean = tf.reduce_mean(mfcc)
        std = tf.math.reduce_std(mfcc)
        mfcc = (mfcc - mean) / (std + 1e-6)

        mfcc = tf.expand_dims(mfcc, axis=-1)

        # time dimension dibuat fleksibel
        mfcc.set_shape([self.n_mfcc, None, 1])

        return mfcc

    @tf.function
    def preprocess_audio(self, file_path, label):
        audio = self.decode_audio(file_path)
        audio = self.fix_audio_length(audio)

        waveform_input = self.make_waveform_input(audio)
        mfcc_input = self.make_mfcc(audio)

        label = tf.cast(label, tf.int32)

        inputs = {
            "waveform_input": waveform_input,
            "mfcc_input": mfcc_input
        }

        return inputs, label

    def create_dataset(self, split, shuffle=False):
        file_paths, labels = self.collect_file_paths(split)

    # Acak file path + label sebelum masuk tf.data
        if shuffle:
            combined = list(zip(file_paths, labels))

            rng = random.Random(42)
            rng.shuffle(combined)

            file_paths, labels = zip(*combined)

            file_paths = list(file_paths)
            labels = list(labels)

        dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))

    # Shuffle ulang setiap epoch
        if shuffle:
            dataset = dataset.shuffle(
                buffer_size=len(file_paths),
                seed=42,
            reshuffle_each_iteration=True
        )

        dataset = dataset.map(
            self.preprocess_audio,
            num_parallel_calls=self.autotune
        )

        dataset = dataset.batch(self.batch_size)
        dataset = dataset.prefetch(self.autotune)

        return dataset

    def get_datasets(self):
        train_ds = self.create_dataset("training", shuffle=True)
        val_ds = self.create_dataset("validation", shuffle=False)
        test_ds = self.create_dataset("testing", shuffle=False)

        return train_ds, val_ds, test_ds

    def preview_batch(self, dataset):
        for inputs, labels in dataset.take(1):
            print("Waveform input shape :", inputs["waveform_input"].shape)
            print("MFCC input shape     :", inputs["mfcc_input"].shape)
            print("Label shape          :", labels.shape)
            print("Label sample         :", labels[:10].numpy())

In [22]:
preprocessor = PreprocessDatasetMFCCTorchLike(
    dataset_root="/kaggle/working/subset_fake_real_dataset_62k",
    sample_rate=16000,
    duration=2.0,
    batch_size=32,
    ignore_errors=False
)


train_ds, val_ds, test_ds = preprocessor.get_datasets()
preprocessor.preview_batch(train_ds)

I0000 00:00:1780520823.149242      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780520823.155198      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


PreprocessDatasetMFCCTorchLike initialized.
Dataset root     : /kaggle/working/subset_fake_real_dataset_62k
Sample rate      : 16000
Duration         : 2.0
Waveform samples : 32000
N MFCC           : 40
N Mels           : 64
Frame length     : 512
Frame step       : 160
FFT length       : 512
Batch size       : 32
training | real: 26082 files
training | fake: 26080 files
Total training: 52162 files

validation | real: 2000 files
validation | fake: 1999 files
Total validation: 3999 files

testing | real: 1000 files
testing | fake: 1000 files
Total testing: 2000 files

Waveform input shape : (32, 32000, 1)
MFCC input shape     : (32, 40, 201, 1)
Label shape          : (32,)
Label sample         : [1 0 1 1 0 0 1 1 0 0]


In [23]:
for inputs, labels in train_ds.take(1):
    print(inputs["waveform_input"].shape)
    print(inputs["mfcc_input"].shape)
    print(labels.shape)

(32, 32000, 1)
(32, 40, 201, 1)
(32,)


In [24]:
# mel_input = tf.keras.Input(shape=(128, 128, 1), name="mel_input")
# waveform_input = tf.keras.Input(shape=(32000, 1), name="waveform_input") -> contoh bikin modelnya nnti

In [25]:
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10
# ) -> Contoh train model (mirip2 sih)

In [26]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from pathlib import Path
import datetime

In [27]:
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name="binary_focal_loss"):
        super().__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_true = tf.reshape(y_true, (-1, 1))

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))

        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_factor = y_true * self.alpha + (1.0 - y_true) * (1.0 - self.alpha)
        modulating_factor = tf.pow(1.0 - p_t, self.gamma)

        loss = alpha_factor * modulating_factor * bce
        return tf.reduce_mean(loss)

In [28]:
class AdaptiveAvgPool1D(tf.keras.layers.Layer):
    def __init__(self, output_size, **kwargs):
        super().__init__(**kwargs)
        self.output_size = output_size

    def call(self, inputs):
        # inputs: (B, T, C)
        x = tf.transpose(inputs, [0, 2, 1])      # (B, C, T)
        x = tf.expand_dims(x, axis=-1)           # (B, C, T, 1)

        x = tf.image.resize(
            x,
            size=[tf.shape(x)[1], self.output_size],
            method="bilinear"
        )

        x = tf.squeeze(x, axis=-1)               # (B, C, output_size)
        x = tf.transpose(x, [0, 2, 1])           # (B, output_size, C)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_size": self.output_size
        })
        return config


class AdaptiveAvgPool2D(tf.keras.layers.Layer):
    def __init__(self, output_size, **kwargs):
        super().__init__(**kwargs)
        self.output_size = output_size

    def call(self, inputs):
        # inputs: (B, H, W, C)
        return tf.image.resize(
            inputs,
            size=self.output_size,
            method="bilinear"
        )

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_size": self.output_size
        })
        return config

In [29]:
class WeightedFusionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.fusion_logits = self.add_weight(
            name="fusion_logits",
            shape=(2,),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        mfcc_features, wave_features = inputs

        weights = tf.nn.softmax(self.fusion_logits)

        mfcc_weighted = mfcc_features * weights[0]
        wave_weighted = wave_features * weights[1]

        fused = tf.concat([mfcc_weighted, wave_weighted], axis=-1)

        return fused

    def compute_output_shape(self, input_shape):
        mfcc_shape, wave_shape = input_shape
        return (mfcc_shape[0], mfcc_shape[-1] + wave_shape[-1])

    def get_config(self):
        config = super().get_config()
        return config

In [30]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [31]:
def build_waveform_branch_torchlike(waveform_input):
    x = layers.Conv1D(32, kernel_size=5, strides=2, padding="same")(waveform_input)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv1D(64, kernel_size=5, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv1D(128, kernel_size=5, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Mirip nn.AdaptiveAvgPool1d(32)
    x = AdaptiveAvgPool1D(32, name="adaptive_avg_pool_1d")(x)

    x = layers.Flatten()(x)

    return x

In [32]:
def build_mfcc_branch_torchlike(mfcc_input):
    x = layers.Conv2D(32, kernel_size=3, padding="same")(mfcc_input)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, kernel_size=3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.Conv2D(128, kernel_size=3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Mirip nn.AdaptiveAvgPool2d((8, 8))
    x = AdaptiveAvgPool2D((8, 8), name="adaptive_avg_pool_2d")(x)

    x = layers.Flatten()(x)

    return x

In [33]:
def build_hybrid_audio_cnn_torchlike(
    waveform_shape=(32000, 1),
    mfcc_shape=(40, None, 1),
    num_classes=2
):
    waveform_input = layers.Input(
        shape=waveform_shape,
        name="waveform_input"
    )

    mfcc_input = layers.Input(
        shape=mfcc_shape,
        name="mfcc_input"
    )

    wave_features = build_waveform_branch_torchlike(waveform_input)
    mfcc_features = build_mfcc_branch_torchlike(mfcc_input)

    x = layers.Concatenate()([wave_features, mfcc_features])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    logits = layers.Dense(num_classes, name="logits")(x)

    model = Model(
        inputs={
            "waveform_input": waveform_input,
            "mfcc_input": mfcc_input
        },
        outputs=logits,
        name="TorchLike_MFCC_Waveform_HybridCNN"
    )

    return model

In [34]:
model = build_hybrid_audio_cnn_torchlike(
    waveform_shape=(32000, 1),
    mfcc_shape=(40, None, 1),
    num_classes=2
)

model.summary()
print(f"Total params: {model.count_params() / 1e6:.2f}M")

Model: "TorchLike_MFCC_Waveform_HybridCNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mfcc_input          │ (None, 40, None,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ waveform_input      │ (None, 32000, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 40, None,  │        320 │ mfcc_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 16000, 32) │        192 │ waveform_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 40, None,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16000, 32) │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 40, None,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 16000, 32) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 40, None,  │     18,496 │ re_lu_3[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 8000, 64)  │     10,304 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 40, None,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8000, 64)  │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 40, None,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 8000, 64)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 20, None,  │          0 │ re_lu_4[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 4000, 128) │     41,088 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 20, None,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4000, 128) │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 3,292,546 (12.56 MB)

 Trainable params: 3,291,650 (12.56 MB)

 Non-trainable params: 896 (3.50 KB)

Total params: 3.29M


In [35]:
total_params = model.count_params()
print(f"Total params: {total_params / 1e6:.2f}M")

Total params: 3.29M


In [36]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

train_loss_metric = tf.keras.metrics.Mean(name="train_loss")
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="train_accuracy")
train_precision = tf.keras.metrics.Precision(name="train_precision")
train_recall = tf.keras.metrics.Recall(name="train_recall")
train_mae = tf.keras.metrics.MeanAbsoluteError(name="train_mae")

val_loss_metric = tf.keras.metrics.Mean(name="val_loss")
val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="val_accuracy")
val_precision = tf.keras.metrics.Precision(name="val_precision")
val_recall = tf.keras.metrics.Recall(name="val_recall")
val_mae = tf.keras.metrics.MeanAbsoluteError(name="val_mae")

In [37]:
def reset_train_metrics():
    train_loss_metric.reset_state()
    train_accuracy.reset_state()
    train_precision.reset_state()
    train_recall.reset_state()
    train_mae.reset_state()


def reset_val_metrics():
    val_loss_metric.reset_state()
    val_accuracy.reset_state()
    val_precision.reset_state()
    val_recall.reset_state()
    val_mae.reset_state()

In [38]:
@tf.function
def train_step(inputs, labels):
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:
        logits = model(inputs, training=True)
        loss = loss_fn(labels, logits)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    probs = tf.nn.softmax(logits, axis=-1)
    pred_classes = tf.argmax(probs, axis=-1, output_type=tf.int32)

    fake_probs = probs[:, 1]
    labels_float = tf.cast(labels, tf.float32)

    train_loss_metric.update_state(loss)
    train_accuracy.update_state(labels, logits)
    train_precision.update_state(labels, pred_classes)
    train_recall.update_state(labels, pred_classes)
    train_mae.update_state(labels_float, fake_probs)

    return loss

In [39]:
@tf.function
def val_step(inputs, labels):
    labels = tf.cast(labels, tf.int32)

    logits = model(inputs, training=False)
    loss = loss_fn(labels, logits)

    probs = tf.nn.softmax(logits, axis=-1)
    pred_classes = tf.argmax(probs, axis=-1, output_type=tf.int32)

    fake_probs = probs[:, 1]
    labels_float = tf.cast(labels, tf.float32)

    val_loss_metric.update_state(loss)
    val_accuracy.update_state(labels, logits)
    val_precision.update_state(labels, pred_classes)
    val_recall.update_state(labels, pred_classes)
    val_mae.update_state(labels_float, fake_probs)

    return loss

In [40]:
log_dir = Path("/kaggle/working/logs") / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_summary_writer = tf.summary.create_file_writer(str(log_dir / "train"))
val_summary_writer = tf.summary.create_file_writer(str(log_dir / "val"))

print(f"TensorBoard log dir: {log_dir}")

TensorBoard log dir: /kaggle/working/logs/20260603-210710


In [41]:
import shutil
from pathlib import Path

import tensorflow as tf


def train_model_custom_loop(
    model,
    train_ds,
    val_ds,
    train_summary_writer,
    val_summary_writer,
    epochs=5,
    keras_save_path="/kaggle/working/best_hybrid_audio_model.keras",
    savedmodel_export_path="/kaggle/working/best_hybrid_audio_savedmodel"
):
    best_val_accuracy = 0.0

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_precision": [],
        "train_recall": [],
        "train_f1": [],
        "train_mae": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [],
        "val_mae": []
    }

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        print("-" * 70)

        # ====================================================
        # RESET METRICS
        # ====================================================

        reset_train_metrics()
        reset_val_metrics()

        # ====================================================
        # TRAINING LOOP
        # ====================================================

        for step, (inputs, labels) in enumerate(train_ds):
            train_step(
                inputs,
                labels
            )

            if step % 100 == 0:
                print(
                    f"Step {step} | "
                    f"Loss: {train_loss_metric.result():.4f} | "
                    f"Acc: {train_accuracy.result():.4f} | "
                    f"MAE: {train_mae.result():.4f}"
                )

        # ====================================================
        # VALIDATION LOOP
        # ====================================================

        for inputs, labels in val_ds:
            val_step(
                inputs,
                labels
            )

        # ====================================================
        # CALCULATE F1 SCORE
        # ====================================================

        train_f1 = (
            2
            * train_precision.result()
            * train_recall.result()
            / (
                train_precision.result()
                + train_recall.result()
                + 1e-8
            )
        )

        val_f1 = (
            2
            * val_precision.result()
            * val_recall.result()
            / (
                val_precision.result()
                + val_recall.result()
                + 1e-8
            )
        )

        # ====================================================
        # GET RESULTS
        # ====================================================

        train_loss_value = float(
            train_loss_metric.result().numpy()
        )

        train_accuracy_value = float(
            train_accuracy.result().numpy()
        )

        train_precision_value = float(
            train_precision.result().numpy()
        )

        train_recall_value = float(
            train_recall.result().numpy()
        )

        train_f1_value = float(
            train_f1.numpy()
        )

        train_mae_value = float(
            train_mae.result().numpy()
        )

        val_loss_value = float(
            val_loss_metric.result().numpy()
        )

        val_accuracy_value = float(
            val_accuracy.result().numpy()
        )

        val_precision_value = float(
            val_precision.result().numpy()
        )

        val_recall_value = float(
            val_recall.result().numpy()
        )

        val_f1_value = float(
            val_f1.numpy()
        )

        val_mae_value = float(
            val_mae.result().numpy()
        )

        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print(
            f"\nTrain Loss: {train_loss_value:.4f} | "
            f"Train Acc: {train_accuracy_value:.4f} | "
            f"Train Precision: {train_precision_value:.4f} | "
            f"Train Recall: {train_recall_value:.4f} | "
            f"Train F1: {train_f1_value:.4f} | "
            f"Train MAE: {train_mae_value:.4f}"
        )

        print(
            f"Val Loss: {val_loss_value:.4f} | "
            f"Val Acc: {val_accuracy_value:.4f} | "
            f"Val Precision: {val_precision_value:.4f} | "
            f"Val Recall: {val_recall_value:.4f} | "
            f"Val F1: {val_f1_value:.4f} | "
            f"Val MAE: {val_mae_value:.4f}"
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history["train_loss"].append(
            train_loss_value
        )

        history["train_accuracy"].append(
            train_accuracy_value
        )

        history["train_precision"].append(
            train_precision_value
        )

        history["train_recall"].append(
            train_recall_value
        )

        history["train_f1"].append(
            train_f1_value
        )

        history["train_mae"].append(
            train_mae_value
        )

        history["val_loss"].append(
            val_loss_value
        )

        history["val_accuracy"].append(
            val_accuracy_value
        )

        history["val_precision"].append(
            val_precision_value
        )

        history["val_recall"].append(
            val_recall_value
        )

        history["val_f1"].append(
            val_f1_value
        )

        history["val_mae"].append(
            val_mae_value
        )

        # ====================================================
        # WRITE TENSORBOARD LOGS
        # ====================================================

        with train_summary_writer.as_default():
            tf.summary.scalar(
                "loss",
                train_loss_value,
                step=epoch
            )

            tf.summary.scalar(
                "accuracy",
                train_accuracy_value,
                step=epoch
            )

            tf.summary.scalar(
                "precision",
                train_precision_value,
                step=epoch
            )

            tf.summary.scalar(
                "recall",
                train_recall_value,
                step=epoch
            )

            tf.summary.scalar(
                "f1_score",
                train_f1_value,
                step=epoch
            )

            tf.summary.scalar(
                "mae",
                train_mae_value,
                step=epoch
            )

        with val_summary_writer.as_default():
            tf.summary.scalar(
                "loss",
                val_loss_value,
                step=epoch
            )

            tf.summary.scalar(
                "accuracy",
                val_accuracy_value,
                step=epoch
            )

            tf.summary.scalar(
                "precision",
                val_precision_value,
                step=epoch
            )

            tf.summary.scalar(
                "recall",
                val_recall_value,
                step=epoch
            )

            tf.summary.scalar(
                "f1_score",
                val_f1_value,
                step=epoch
            )

            tf.summary.scalar(
                "mae",
                val_mae_value,
                step=epoch
            )

        # Wajib agar scalar langsung masuk event file
        train_summary_writer.flush()
        val_summary_writer.flush()

        print("TensorBoard logs updated.")

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_accuracy_value > best_val_accuracy:
            best_val_accuracy = val_accuracy_value

            model.save(
                keras_save_path
            )

            print(
                f"Best .keras model saved to: "
                f"{keras_save_path}"
            )

            savedmodel_path = Path(
                savedmodel_export_path
            )

            # Hapus export lama agar tidak bentrok
            if savedmodel_path.exists():
                shutil.rmtree(
                    savedmodel_path
                )

            model.export(
                savedmodel_export_path
            )

            print(
                f"Best SavedModel exported to: "
                f"{savedmodel_export_path}"
            )

    print("\nTraining finished.")

    print(
        f"Best validation accuracy: "
        f"{best_val_accuracy:.4f}"
    )

    return model, history

In [42]:
import os

for root, dirs, files in os.walk("/kaggle/working/logs"):
    for file in files:
        if "tfevents" in file:
            print(os.path.join(root, file))

/kaggle/working/logs/20260603-210710/val/events.out.tfevents.1780520830.82e2e179a54d.58.1.v2
/kaggle/working/logs/20260603-210710/train/events.out.tfevents.1780520830.82e2e179a54d.58.0.v2


In [43]:
model, history = train_model_custom_loop(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    train_summary_writer=train_summary_writer,
    val_summary_writer=val_summary_writer,
    epochs=5,
    keras_save_path=(
        "/kaggle/working/"
        "best_hybrid_audio_model.keras"
    ),
    savedmodel_export_path=(
        "/kaggle/working/"
        "best_hybrid_audio_savedmodel"
    )
)


Epoch 1/5
----------------------------------------------------------------------


I0000 00:00:1780520833.969816     139 cuda_dnn.cc:529] Loaded cuDNN version 91002


Step 0 | Loss: 1.5126 | Acc: 0.3438 | MAE: 0.6264
Step 100 | Loss: 0.7210 | Acc: 0.6139 | MAE: 0.4376
Step 200 | Loss: 0.6655 | Acc: 0.6497 | MAE: 0.4142
Step 300 | Loss: 0.6300 | Acc: 0.6720 | MAE: 0.3991
Step 400 | Loss: 0.6024 | Acc: 0.6894 | MAE: 0.3860
Step 500 | Loss: 0.5827 | Acc: 0.7033 | MAE: 0.3751
Step 600 | Loss: 0.5664 | Acc: 0.7151 | MAE: 0.3664
Step 700 | Loss: 0.5521 | Acc: 0.7248 | MAE: 0.3582
Step 800 | Loss: 0.5396 | Acc: 0.7335 | MAE: 0.3508
Step 900 | Loss: 0.5258 | Acc: 0.7423 | MAE: 0.3429
Step 1000 | Loss: 0.5142 | Acc: 0.7495 | MAE: 0.3358
Step 1100 | Loss: 0.5022 | Acc: 0.7567 | MAE: 0.3288
Step 1200 | Loss: 0.4909 | Acc: 0.7637 | MAE: 0.3217
Step 1300 | Loss: 0.4814 | Acc: 0.7694 | MAE: 0.3157
Step 1400 | Loss: 0.4719 | Acc: 0.7752 | MAE: 0.3097
Step 1500 | Loss: 0.4624 | Acc: 0.7815 | MAE: 0.3034
Step 1600 | Loss: 0.4533 | Acc: 0.7866 | MAE: 0.2976

Train Loss: 0.4509 | Train Acc: 0.7880 | Train Precision: 0.7927 | Train Recall: 0.7800 | Train F1: 0.7863 | T

INFO:tensorflow:Assets written to: /kaggle/working/best_hybrid_audio_savedmodel/assets


Saved artifact at '/kaggle/working/best_hybrid_audio_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['waveform_input', TensorSpec(shape=(None, 32000, 1), dtype=tf.float32, name='waveform_input')], ['mfcc_input', TensorSpec(shape=(None, 40, None, 1), dtype=tf.float32, name='mfcc_input')]]
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140568686243536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686246032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686245072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686247184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691520720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691523408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691

...

Saved artifact at '/kaggle/working/best_hybrid_audio_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['waveform_input', TensorSpec(shape=(None, 32000, 1), dtype=tf.float32, name='waveform_input')], ['mfcc_input', TensorSpec(shape=(None, 40, None, 1), dtype=tf.float32, name='mfcc_input')]]
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140568686243536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686246032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686245072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686247184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691520720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691523408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691

INFO:tensorflow:Assets written to: /kaggle/working/best_hybrid_audio_savedmodel/assets


Saved artifact at '/kaggle/working/best_hybrid_audio_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['waveform_input', TensorSpec(shape=(None, 32000, 1), dtype=tf.float32, name='waveform_input')], ['mfcc_input', TensorSpec(shape=(None, 40, None, 1), dtype=tf.float32, name='mfcc_input')]]
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140568686243536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686246032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686245072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686247184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691520720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691523408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691

INFO:tensorflow:Assets written to: /kaggle/working/best_hybrid_audio_savedmodel/assets


Saved artifact at '/kaggle/working/best_hybrid_audio_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['waveform_input', TensorSpec(shape=(None, 32000, 1), dtype=tf.float32, name='waveform_input')], ['mfcc_input', TensorSpec(shape=(None, 40, None, 1), dtype=tf.float32, name='mfcc_input')]]
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140568686243536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686246032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686241424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686245072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568686247184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691520720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691523408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140568691

In [44]:

test_loss_metric = tf.keras.metrics.Mean(name="test_loss")
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="test_accuracy")
test_precision = tf.keras.metrics.Precision(name="test_precision")
test_recall = tf.keras.metrics.Recall(name="test_recall")
test_mae = tf.keras.metrics.MeanAbsoluteError(name="test_mae")


@tf.function
def test_step_best(inputs, labels):
    labels = tf.cast(labels, tf.int32)

    logits = best_model(inputs, training=False)
    loss = loss_fn(labels, logits)

    probs = tf.nn.softmax(logits, axis=-1)
    pred_classes = tf.argmax(probs, axis=-1, output_type=tf.int32)

    fake_probs = probs[:, 1]
    labels_float = tf.cast(labels, tf.float32)

    test_loss_metric.update_state(loss)
    test_accuracy.update_state(labels, logits)
    test_precision.update_state(labels, pred_classes)
    test_recall.update_state(labels, pred_classes)
    test_mae.update_state(labels_float, fake_probs)

    return pred_classes

In [45]:
def evaluate_best_model(test_ds):
    test_loss_metric.reset_state()
    test_accuracy.reset_state()
    test_precision.reset_state()
    test_recall.reset_state()
    test_mae.reset_state()

    for inputs, labels in test_ds:
        test_step_best(inputs, labels)

    test_f1 = 2 * (
        test_precision.result() * test_recall.result()
    ) / (
        test_precision.result() + test_recall.result() + 1e-8
    )

    print("\nBest Model Test Result")
    print("=" * 40)
    print(f"Test Loss      : {test_loss_metric.result():.4f}")
    print(f"Test Accuracy  : {test_accuracy.result():.4f}")
    print(f"Test Precision : {test_precision.result():.4f}")
    print(f"Test Recall    : {test_recall.result():.4f}")
    print(f"Test F1-score  : {test_f1:.4f}")
    print(f"Test MAE       : {test_mae.result():.4f}")

In [46]:
!find /kaggle/working -name "*.keras"

/kaggle/working/best_hybrid_audio_model.keras


In [47]:
best_model = tf.keras.models.load_model(
    "/kaggle/working/best_hybrid_audio_model.keras",
    custom_objects={
        "AdaptiveAvgPool1D": AdaptiveAvgPool1D,
        "AdaptiveAvgPool2D": AdaptiveAvgPool2D,
        "BinaryFocalLoss": BinaryFocalLoss,
    }
)

In [48]:
evaluate_best_model(test_ds)


Best Model Test Result
Test Loss      : 0.3651
Test Accuracy  : 0.8380
Test Precision : 0.7780
Test Recall    : 0.9460
Test F1-score  : 0.8538
Test MAE       : 0.1976


In [49]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

In [50]:
def get_predictions_and_labels(model, dataset, name="dataset"):
    all_fake_probs = []
    all_preds = []
    all_labels = []

    for inputs, labels in dataset:
        logits = model(inputs, training=False)

        # Karena model output-nya 2 logits
        probs = tf.nn.softmax(logits, axis=-1).numpy()

        # Probabilitas kelas fake = index 1
        fake_probs = probs[:, 1]

        # Prediksi class: 0=real, 1=fake
        preds = np.argmax(probs, axis=1)

        labels = labels.numpy().astype(int)

        all_fake_probs.extend(fake_probs)
        all_preds.extend(preds)
        all_labels.extend(labels)

    all_fake_probs = np.array(all_fake_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    print(f"\n{name} prediction distribution")
    print("=" * 50)
    print(f"Fake prob min  : {all_fake_probs.min():.4f}")
    print(f"Fake prob max  : {all_fake_probs.max():.4f}")
    print(f"Fake prob mean : {all_fake_probs.mean():.4f}")
    print(f"Fake prob std  : {all_fake_probs.std():.4f}")

    print("\nPredicted labels:")
    print(f"Pred real 0 : {(all_preds == 0).sum()}")
    print(f"Pred fake 1 : {(all_preds == 1).sum()}")

    print("\nTrue labels:")
    print(f"True real 0 : {(all_labels == 0).sum()}")
    print(f"True fake 1 : {(all_labels == 1).sum()}")

    return all_fake_probs, all_preds, all_labels

In [51]:
train_probs, train_preds, train_labels = get_predictions_and_labels(
    best_model,
    train_ds,
    name="train"
)

val_probs, val_preds, val_labels = get_predictions_and_labels(
    best_model,
    val_ds,
    name="validation"
)

test_probs, test_preds, test_labels = get_predictions_and_labels(
    best_model,
    test_ds,
    name="test"
)


train prediction distribution
Fake prob min  : 0.0000
Fake prob max  : 1.0000
Fake prob mean : 0.5018
Fake prob std  : 0.4662

Predicted labels:
Pred real 0 : 25738
Pred fake 1 : 26424

True labels:
True real 0 : 26082
True fake 1 : 26080

validation prediction distribution
Fake prob min  : 0.0000
Fake prob max  : 1.0000
Fake prob mean : 0.4971
Fake prob std  : 0.4599

Predicted labels:
Pred real 0 : 1994
Pred fake 1 : 2005

True labels:
True real 0 : 2000
True fake 1 : 1999

test prediction distribution
Fake prob min  : 0.0001
Fake prob max  : 1.0000
Fake prob mean : 0.6102
Fake prob std  : 0.3946

Predicted labels:
Pred real 0 : 784
Pred fake 1 : 1216

True labels:
True real 0 : 1000
True fake 1 : 1000


In [52]:
from sklearn.metrics import confusion_matrix, classification_report

def show_report(labels, preds, name="dataset"):
    print(f"\n{name} confusion matrix")
    print("=" * 50)
    print(confusion_matrix(labels, preds))

    print(f"\n{name} classification report")
    print("=" * 50)
    print(classification_report(
        labels,
        preds,
        target_names=["real", "fake"],
        zero_division=0
    ))

In [53]:
show_report(train_labels, train_preds, "train")
show_report(val_labels, val_preds, "validation")
show_report(test_labels, test_preds, "test")


train confusion matrix
[[25392   690]
 [  346 25734]]

train classification report
              precision    recall  f1-score   support

        real       0.99      0.97      0.98     26082
        fake       0.97      0.99      0.98     26080

    accuracy                           0.98     52162
   macro avg       0.98      0.98      0.98     52162
weighted avg       0.98      0.98      0.98     52162


validation confusion matrix
[[1909   91]
 [  85 1914]]

validation classification report
              precision    recall  f1-score   support

        real       0.96      0.95      0.96      2000
        fake       0.95      0.96      0.96      1999

    accuracy                           0.96      3999
   macro avg       0.96      0.96      0.96      3999
weighted avg       0.96      0.96      0.96      3999


test confusion matrix
[[730 270]
 [ 54 946]]

test classification report
              precision    recall  f1-score   support

        real       0.93      0.73      0.82

In [54]:
import numpy as np
import librosa
import tensorflow as tf


def preprocess_single_audio(
    file_path,
    sample_rate=16000,
    duration=2.0,
    n_mfcc=40,
    n_mels=64,
    frame_length=512,
    frame_step=160,
    fft_length=512
):
    """
    Load dan preprocess satu audio agar format input-nya sama
    dengan pipeline training.

    Output:
        {
            "waveform_input": shape (1, 32000, 1),
            "mfcc_input": shape (1, 40, time_frames, 1)
        }
    """

    num_samples = int(sample_rate * duration)

    # Load audio dan resample otomatis ke 16 kHz
    audio, _ = librosa.load(
        file_path,
        sr=sample_rate,
        mono=True
    )

    audio = audio.astype(np.float32)

    # Potong atau padding hingga tepat 2 detik
    if len(audio) > num_samples:
        audio = audio[:num_samples]

    elif len(audio) < num_samples:
        padding = num_samples - len(audio)
        audio = np.pad(
            audio,
            pad_width=(0, padding),
            mode="constant"
        )

    audio = tf.convert_to_tensor(
        audio,
        dtype=tf.float32
    )

    # Input cabang waveform: (batch, samples, channel)
    waveform_input = tf.expand_dims(audio, axis=-1)
    waveform_input = tf.expand_dims(waveform_input, axis=0)

    # Dibuat sama dengan preprocessing training:
    # center=True versi manual melalui padding
    pad = fft_length // 2

    audio_centered = tf.pad(
        audio,
        paddings=[[pad, pad]]
    )

    stft = tf.signal.stft(
        audio_centered,
        frame_length=frame_length,
        frame_step=frame_step,
        fft_length=fft_length
    )

    spectrogram = tf.abs(stft)
    power_spectrogram = tf.square(spectrogram)

    num_spectrogram_bins = fft_length // 2 + 1

    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=n_mels,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=sample_rate,
        lower_edge_hertz=80.0,
        upper_edge_hertz=7600.0
    )

    mel_spectrogram = tf.matmul(
        power_spectrogram,
        mel_weight_matrix
    )

    log_mel_spectrogram = tf.math.log(
        mel_spectrogram + 1e-6
    )

    mfcc = tf.signal.mfccs_from_log_mel_spectrograms(
        log_mel_spectrogram
    )

    # Ambil 40 koefisien MFCC
    mfcc = mfcc[:, :n_mfcc]

    # Ubah dari (time, mfcc) menjadi (mfcc, time)
    mfcc = tf.transpose(mfcc)

    # Normalisasi seperti saat training
    mean = tf.reduce_mean(mfcc)
    std = tf.math.reduce_std(mfcc)

    mfcc = (mfcc - mean) / (std + 1e-6)

    # Input cabang MFCC: (batch, mfcc, time, channel)
    mfcc_input = tf.expand_dims(mfcc, axis=-1)
    mfcc_input = tf.expand_dims(mfcc_input, axis=0)

    return {
        "waveform_input": waveform_input,
        "mfcc_input": mfcc_input
    }

In [55]:
def predict_audio(
    model,
    file_path
):
    """
    Melakukan inference satu file audio.
    Output:
    - predicted_label
    - probability_real
    - probability_fake
    """

    inputs = preprocess_single_audio(
        file_path=file_path
    )

    logits = model(
        inputs,
        training=False
    )

    probabilities = tf.nn.softmax(
        logits,
        axis=-1
    ).numpy()[0]

    probability_real = float(probabilities[0])
    probability_fake = float(probabilities[1])

    if probability_fake >= probability_real:
        predicted_label = "fake"
    else:
        predicted_label = "real"

    result = {
        "prediction": predicted_label,
        "probability_real": probability_real,
        "probability_fake": probability_fake
    }

    return result

In [56]:
audio_path = (r"/kaggle/working/subset_fake_real_dataset_62k/testing/fake/for-2sec__file1018.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav")

result = predict_audio(
    model=best_model,
    file_path=audio_path
)

print(result)

{'prediction': 'fake', 'probability_real': 0.0036136468406766653, 'probability_fake': 0.9963864088058472}


In [57]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error
)


def evaluate_with_threshold(
    model,
    dataset,
    threshold=0.60
):
    """
    Evaluasi dataset menggunakan threshold probability fake.

    class 0 = real
    class 1 = fake

    Prediksi:
        fake jika probability_fake >= threshold
        real jika probability_fake < threshold
    """

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    for batch_inputs, batch_labels in dataset:
        logits = model(
            batch_inputs,
            training=False
        )

        probabilities = tf.nn.softmax(
            logits,
            axis=-1
        )

        fake_probabilities = probabilities[:, 1]

        batch_predictions = tf.cast(
            fake_probabilities >= threshold,
            dtype=tf.int32
        )

        all_labels.extend(
            batch_labels.numpy().tolist()
        )

        all_predictions.extend(
            batch_predictions.numpy().tolist()
        )

        all_fake_probabilities.extend(
            fake_probabilities.numpy().tolist()
        )

    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)
    y_fake_prob = np.array(all_fake_probabilities)

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    # MAE menggunakan probabilitas fake dan label ground truth 0 atau 1
    mae = mean_absolute_error(
        y_true,
        y_fake_prob
    )

    print("=" * 55)
    print(f"TEST EVALUATION — THRESHOLD = {threshold:.2f}")
    print("=" * 55)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nInterpretasi Confusion Matrix:")
    print(f"Real diprediksi Real : {cm[0, 0]}")
    print(f"Real diprediksi Fake : {cm[0, 1]}")
    print(f"Fake diprediksi Real : {cm[1, 0]}")
    print(f"Fake diprediksi Fake : {cm[1, 1]}")

    print("\nMetrics:")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")
    print(f"MAE       : {mae:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["real", "fake"],
            digits=4,
            zero_division=0
        )
    )

    return {
        "threshold": threshold,
        "confusion_matrix": cm,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "macro_f1": macro_f1,
        "mae": mae,
        "y_true": y_true,
        "y_pred": y_pred,
        "fake_probabilities": y_fake_prob
    }

In [58]:
test_result_threshold_050 = evaluate_with_threshold(
    model=best_model,
    dataset=test_ds,
    threshold=0.50
)

test_result_threshold_060 = evaluate_with_threshold(
    model=best_model,
    dataset=test_ds,
    threshold=0.60
)

test_result_threshold_060 = evaluate_with_threshold(
    model=best_model,
    dataset=test_ds,
    threshold=0.70
)

TEST EVALUATION — THRESHOLD = 0.50

Confusion Matrix:
[[730 270]
 [ 54 946]]

Interpretasi Confusion Matrix:
Real diprediksi Real : 730
Real diprediksi Fake : 270
Fake diprediksi Real : 54
Fake diprediksi Fake : 946

Metrics:
Accuracy  : 0.8380
Precision : 0.7780
Recall    : 0.9460
F1-score  : 0.8538
Macro F1  : 0.8361
MAE       : 0.1973

Classification Report:
              precision    recall  f1-score   support

        real     0.9311    0.7300    0.8184      1000
        fake     0.7780    0.9460    0.8538      1000

    accuracy                         0.8380      2000
   macro avg     0.8545    0.8380    0.8361      2000
weighted avg     0.8545    0.8380    0.8361      2000

TEST EVALUATION — THRESHOLD = 0.60

Confusion Matrix:
[[774 226]
 [ 71 929]]

Interpretasi Confusion Matrix:
Real diprediksi Real : 774
Real diprediksi Fake : 226
Fake diprediksi Real : 71
Fake diprediksi Fake : 929

Metrics:
Accuracy  : 0.8515
Precision : 0.8043
Recall    : 0.9290
F1-score  : 0.8622
Macro F

In [59]:
from pathlib import Path

LOG_ROOT = Path("/kaggle/working/logs")

if LOG_ROOT.exists():
    print("Folder log ditemukan:")
    
    for path in sorted(LOG_ROOT.rglob("*")):
        print(path)

else:
    print("Folder log belum ditemukan:", LOG_ROOT)

Folder log ditemukan:
/kaggle/working/logs/20260603-210710
/kaggle/working/logs/20260603-210710/train
/kaggle/working/logs/20260603-210710/train/events.out.tfevents.1780520830.82e2e179a54d.58.0.v2
/kaggle/working/logs/20260603-210710/val
/kaggle/working/logs/20260603-210710/val/events.out.tfevents.1780520830.82e2e179a54d.58.1.v2


In [60]:
import datetime
import tensorflow as tf
from pathlib import Path


# Folder utama seluruh eksperimen
LOG_ROOT = Path("/kaggle/working/logs")

# Folder khusus training saat ini
log_dir = (
    LOG_ROOT
    / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)

train_summary_writer = tf.summary.create_file_writer(
    str(log_dir / "train")
)

val_summary_writer = tf.summary.create_file_writer(
    str(log_dir / "val")
)

print("TensorBoard root directory :", LOG_ROOT)
print("Current training log dir   :", log_dir)

TensorBoard root directory : /kaggle/working/logs
Current training log dir   : /kaggle/working/logs/20260603-212732


In [61]:
!kill 666
!kill 624

/bin/bash: line 1: kill: (666) - No such process
/bin/bash: line 1: kill: (624) - No such process


In [62]:
import os

for root, dirs, files in os.walk("/kaggle/working/logs"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/logs/20260603-210710/val/events.out.tfevents.1780520830.82e2e179a54d.58.1.v2
/kaggle/working/logs/20260603-210710/train/events.out.tfevents.1780520830.82e2e179a54d.58.0.v2
/kaggle/working/logs/20260603-212732/val/events.out.tfevents.1780522052.82e2e179a54d.58.3.v2
/kaggle/working/logs/20260603-212732/train/events.out.tfevents.1780522052.82e2e179a54d.58.2.v2


In [63]:
!find /kaggle/working/logs -name "*tfevents*"

/kaggle/working/logs/20260603-210710/val/events.out.tfevents.1780520830.82e2e179a54d.58.1.v2
/kaggle/working/logs/20260603-210710/train/events.out.tfevents.1780520830.82e2e179a54d.58.0.v2
/kaggle/working/logs/20260603-212732/val/events.out.tfevents.1780522052.82e2e179a54d.58.3.v2
/kaggle/working/logs/20260603-212732/train/events.out.tfevents.1780522052.82e2e179a54d.58.2.v2


In [64]:
from pathlib import Path

import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import (
    EventAccumulator
)


LOG_ROOT = Path("/kaggle/working/logs")

# Ambil folder eksperimen terbaru
run_dirs = sorted(
    [
        path
        for path in LOG_ROOT.iterdir()
        if path.is_dir()
    ]
)

latest_run = run_dirs[-1]

train_dir = latest_run / "train"
val_dir = latest_run / "val"

print("Latest run:", latest_run)


def load_tensorboard_scalars(log_folder):
    event_accumulator = EventAccumulator(
        str(log_folder)
    )

    event_accumulator.Reload()

    scalar_tags = (
        event_accumulator.Tags()
        .get("scalars", [])
    )

    print(
        f"Scalar tags pada {log_folder.name}:",
        scalar_tags
    )

    results = {}

    for tag in scalar_tags:
        events = event_accumulator.Scalars(tag)

        results[tag] = {
            "steps": [
                event.step
                for event in events
            ],
            "values": [
                event.value
                for event in events
            ]
        }

    return results


train_scalars = load_tensorboard_scalars(
    train_dir
)

val_scalars = load_tensorboard_scalars(
    val_dir
)

Latest run: /kaggle/working/logs/20260603-212732
Scalar tags pada train: []
Scalar tags pada val: []


In [65]:
common_tags = sorted(
    set(train_scalars)
    & set(val_scalars)
)

print("Metrics yang akan diplot:", common_tags)


for tag in common_tags:
    plt.figure(figsize=(8, 5))

    plt.plot(
        train_scalars[tag]["steps"],
        train_scalars[tag]["values"],
        marker="o",
        label="Train"
    )

    plt.plot(
        val_scalars[tag]["steps"],
        val_scalars[tag]["values"],
        marker="o",
        label="Validation"
    )

    plt.title(
        f"Training and Validation {tag.capitalize()}"
    )

    plt.xlabel("Epoch")
    plt.ylabel(tag.capitalize())

    plt.legend()
    plt.grid(True)
    plt.show()

Metrics yang akan diplot: []


In [66]:
%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/logs --port 6006

<IPython.core.display.Javascript object>

In [67]:
import zipfile
import os

zip_path = "/kaggle/working/CAPSTONE_SUBMISSION.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    # Model keras
    if os.path.exists(
        "/kaggle/working/best_hybrid_audio_model.keras"
    ):
        zipf.write(
            "/kaggle/working/best_hybrid_audio_model.keras",
            arcname="best_hybrid_audio_model.keras"
        )

    # SavedModel
    for root, dirs, files in os.walk(
        "/kaggle/working/best_hybrid_audio_savedmodel"
    ):
        for file in files:
            filepath = os.path.join(root, file)

            arcname = os.path.relpath(
                filepath,
                "/kaggle/working"
            )

            zipf.write(
                filepath,
                arcname
            )

    # Logs TensorBoard
    if os.path.exists("/kaggle/working/logs"):

        for root, dirs, files in os.walk(
            "/kaggle/working/logs"
        ):
            for file in files:

                filepath = os.path.join(
                    root,
                    file
                )

                arcname = os.path.relpath(
                    filepath,
                    "/kaggle/working"
                )

                zipf.write(
                    filepath,
                    arcname
                )

print("ZIP berhasil dibuat:")
print(zip_path)

ZIP berhasil dibuat:
/kaggle/working/CAPSTONE_SUBMISSION.zip
